# Trabajo Práctico N° 1: Redes Neuronales CBOW para PLN
**Asignatura**: Aprendizaje Automático Avanzado  
**Carrera**: Tecnicatura Universitaria en Inteligencia Artificial  
**Institución**: Universidad Nacional de Hurlingham (UNAHUR)  
**Profesor**: Juan Miguel Santos (LIDEC)  

---

## Descripción del Proyecto
Este Jupyter Notebook implementa el modelo de representación contextual de palabras **CBOW (Continuous Bag-of-Words)** utilizando **Python y CuPy GPU / PyTorch GPU** para aceleración en placas de video NVIDIA.

La implementación se adhiere estrictamente al material teórico y a la formulación matemática expuesta por la cátedra:
- **Entrada**: Contexto de $C = 2 \times W$ palabras ($W$ a la izquierda, $W$ a la derecha).
- **Capa Oculta**: Activación lineal $h = \frac{1}{C} \sum_{c=1}^C W_{I_c, :}^T$
- **Activación y Salida**: Softmax Completa $O(|V|)$ o **Muestreo Negativo (Negative Sampling)** con sigmoides logísticas binarias sobre la distribución $U^{3/4}$.
- **Actualización de Gradientes**: Vectorizada en VRAM GPU sin bucles secuenciales en Python.
- **Token Especial**: Únicamente `<UNK>`.
- **Evaluación de Similaridad**: Producto Interno por defecto, Coseno opcional.
- **Selección de Vocabulario**: Elección excluyente entre `cantidad_palabras_unicas` O `porcentaje_palabras_unicas` con muestreo aleatorio reproducible (`semilla_aleatoria = 26`).
- **Reanudación por Épocas Adicionales**: Método `entrenar_adicional(epocas_adicionales=N)`.

## Sección 1: Configuración de Parámetros y Entorno

La configuración del proyecto se gestiona de forma interactiva y centralizada mediante el **Panel de Control de Hiperparámetros** a continuación.

Al ejecutar la celda de código, los valores definidos en el diccionario `PARAMETROS_PANEL` se aplicarán al objeto `config` en memoria y **se autoguardarán automáticamente** en el archivo físico [`configuracion.yaml`](configuracion.yaml) en disco, garantizando una experimentación ágil y la sincronización total con scripts futuros o ejecuciones externas.

### Parámetros Principales Configurados:
- **`ruta_corpus`**: Ruta al archivo del corpus de texto (ej. `datos/corpus.txt`).
- **`motor_computo`**: Motor de ejecución acelerada en GPU NVIDIA (`"cupy"` o `"pytorch"`).
- **`criterio_seleccion_vocabulario`**: Criterio excluyente (`"cantidad"` o `"porcentaje"`).
- **`cantidad_palabras_unicas` / `porcentaje_palabras_unicas`**: Tamaño del vocabulario seleccionado al azar con semilla `26`.
- **`muestreo_negativo`**: `True` para Muestreo Negativo $O(K)$, `False` para Softmax Completa $O(|V|)$.
- **`cantidad_muestras_negativas`**: Cantidad $K$ de palabras negativas a muestrear ($5$).
- **`tamanio_ventana`**: Ventana $W$ de palabras a izquierda y derecha ($W = 4$).
- **`dimension_embedding`**: Dimensión de la capa oculta ($N = 100$).
- **`tasa_aprendizaje`**: Tasa de aprendizaje $\eta = 0.025$.
- **`cantidad_epocas`**: Número de épocas a entrenar ($10$).
- **`hacer_respaldo`**: Booleano para activar (`True`) o desactivar (`False`) el autoguardado.
- **`frecuencia_respaldo`**: Autoguardado de checkpoints cada $K$ épocas ($2$). Si es $0$ o `hacer_respaldo=False`, se desactiva.
- **`tamanio_lote`**: Tamaño de mini-lote $B$ para procesamiento vectorizado en GPU ($2048$). Opciones según VRAM: 4-6GB ($1024$-$2048$), 8-12GB ($4096$-$8192$), 16GB+ ($8192$-$16384$).
- **`tipo_similaridad`**: Criterio de similaridad para palabras (`"producto_interno"` o `"coseno"`).

In [1]:
import sys
from pathlib import Path

# Asegurar acceso a los módulos del directorio codigo/
ruta_proyecto = Path.cwd()
if str(ruta_proyecto) not in sys.path:
    sys.path.append(str(ruta_proyecto))

from codigo.configuracion import Configuracion
from codigo.tokenizador import Tokenizador
from codigo.generador_vocabulario import GeneradorVocabulario
from codigo.modelo_cbow_cupy import ModeloCbowCuPy, USAR_CUPY
from codigo.modelo_cbow_pytorch import ModeloCbowPyTorch
from codigo.entrenador_cbow import EntrenadorCbow
from codigo.evaluador_similaridad import EvaluadorSimilaridad

# ==============================================================================
# PANEL DE CONTROL DE HIPERPARÁMETROS (MODELO CBOW)
# Modifique libremente los valores en este diccionario para sus experimentos.
# Al ejecutar esta celda, los cambios se guardarán automáticamente en configuracion.yaml
# ==============================================================================
PARAMETROS_PANEL = {
    # Motor de Cómputo
    "motor_computo": "pytorch",                      # Opciones: "cupy" o "pytorch"
    
    # Corpus y Vocabulario
    "ruta_corpus": "datos/corpus.txt",
    "estrategia_tokenizacion": "palabra",         # Opciones: "palabra" o "bpe"
    "token_desconocido": "<UNK>",
    "criterio_seleccion_vocabulario": "porcentaje",# Opciones: "cantidad" o "porcentaje"
    "cantidad_palabras_unicas": 15000,
    "porcentaje_palabras_unicas": 1.0,
    "frecuencia_minima": 1,
    
    # Hiperparámetros CBOW
    "tamanio_ventana": 4,                        # Ventana W (izquierda/derecha)
    "dimension_embedding": 100,                  # Dimensión N
    "tasa_aprendizaje": 0.2,                  # Tasa de aprendizaje eta
    "cantidad_epocas": 10,                       # Número de épocas
    "tamanio_lote": 2048,                        # Muestras por lote (GPU 4-6GB: 1024-2048 | 8-12GB: 4096-8192 | 16GB+: 8192-16384)
    
    # Muestreo Negativo (Negative Sampling)
    "muestreo_negativo": False,                  # True: K binarias, False: Softmax global
    "cantidad_muestras_negativas": 5,           # Cantidad K de palabras negativas
    
    # Resguardos y Evaluación
    "directorio_respaldos": "respaldos",
    "hacer_respaldo": False,                     # True: activa autoguardado, False: desactiva respaldos
    "frecuencia_respaldo": 0,                    # Autoguardado .npz cada K épocas (0 desactiva)
    "tipo_similaridad": "producto_interno",       # "producto_interno" o "coseno"
    "semilla_aleatoria": 26,
}

# --- MECANISMO DE SINCRONIZACIÓN Y AUTOGUARDADO ---
# 1. Cargar la instancia de configuración centralizada
config = Configuracion("configuracion.yaml")

# 2. Actualizar los atributos del objeto en memoria según el Panel de Control
for clave, valor in PARAMETROS_PANEL.items():
    setattr(config, clave, valor)

# 3. Autoguardar instantáneamente en el archivo físico configuracion.yaml en disco
config.guardar_en_yaml()

# 4. Mostrar reporte de configuración sincronizada
print("=== Panel de Control: Hiperparámetros Sincronizados con configuracion.yaml ===")
diccionario_params = config.a_diccionario()
for clave, valor in diccionario_params.items():
    print(f" - {clave}: {valor}")

print(f"\nAceleración GPU (CuPy disponible): {USAR_CUPY}")

=== Panel de Control: Hiperparámetros Sincronizados con configuracion.yaml ===
 - ruta_corpus: datos/corpus.txt
 - directorio_respaldos: respaldos
 - motor_computo: pytorch
 - estrategia_tokenizacion: palabra
 - token_desconocido: <UNK>
 - criterio_seleccion_vocabulario: porcentaje
 - cantidad_palabras_unicas: 15000
 - porcentaje_palabras_unicas: 1.0
 - frecuencia_minima: 1
 - muestreo_negativo: False
 - cantidad_muestras_negativas: 5
 - tamanio_ventana: 4
 - dimension_embedding: 100
 - tasa_aprendizaje: 0.2
 - cantidad_epocas: 10
 - frecuencia_respaldo: 0
 - tamanio_lote: 2048
 - tipo_similaridad: producto_interno
 - semilla_aleatoria: 26

Aceleración GPU (CuPy disponible): True


## Sección 2: Carga y Preprocesamiento del Corpus
Cargamos el archivo de texto especificado en `ruta_corpus` (`datos/corpus.txt`) y realizamos la tokenización por palabras utilizando expresiones regulares (respetando minúsculas, tildes y caracteres en español).

In [2]:
# Instanciar tokenizador
tokenizador = Tokenizador(estrategia=config.estrategia_tokenizacion)
ruta_corpus = Path(config.ruta_corpus)

print(f"Leyendo y tokenizando corpus desde: {ruta_corpus}...")
tokens_corpus = tokenizador.tokenizar_archivo(ruta_corpus)
print(f"Total de palabras/tokens en el corpus: {len(tokens_corpus):,}")
print(f"Muestra de primeros 20 tokens: {tokens_corpus[:20]}")

Leyendo y tokenizando corpus desde: datos/corpus.txt...
Total de palabras/tokens en el corpus: 916,200
Muestra de primeros 20 tokens: ['los', 'enanos', 'tienen', 'una', 'especie', 'de', 'sexto', 'sentido', 'que', 'les', 'permite', 'reconocerse', 'a', 'primera', 'vista', 'a', 'pesar', 'de', 'lo', 'que']


## Sección 3: Construcción del Vocabulario y Distribución $U^{3/4}$
Construimos el vocabulario seleccionando palabras al azar (semilla 26) y calculamos la distribución de frecuencia unigrama rebalanceada a la potencia $3/4$ ($P(w) \propto f(w)^{0.75}$) para el Muestreo Negativo.

In [3]:
# Instanciar generador de vocabulario
vocabulario = GeneradorVocabulario(token_desconocido=config.token_desconocido)

# Construir vocabulario con criterio excluyente y semilla aleatoria
vocabulario.construir_vocabulario(
    lista_tokens=tokens_corpus,
    criterio_seleccion=config.criterio_seleccion_vocabulario,
    cantidad_palabras_unicas=config.cantidad_palabras_unicas,
    porcentaje_palabras_unicas=config.porcentaje_palabras_unicas,
    frecuencia_minima=config.frecuencia_minima,
    semilla_aleatoria=config.semilla_aleatoria,
)

# Convertir la secuencia de palabras del corpus a una lista de indices enteros
indices_corpus = vocabulario.convertir_tokens_a_indices(tokens_corpus)
print(f"Total de indices procesados: {len(indices_corpus):,}")

# Obtener la distribucion de probabilidad unigrama P(w)^0.75 condicional al Muestreo Negativo
if config.muestreo_negativo:
    distribucion_unigrama = vocabulario.obtener_distribucion_unigrama(exponente=0.75)
    print(f"Muestreo Negativo activado: Distribución unigrama P(w)^0.75 calculada ({len(distribucion_unigrama)} elementos).")
else:
    distribucion_unigrama = None
    print("Muestreo Negativo desactivado (Softmax Completa): Se omite el cálculo de la distribución unigrama P(w)^0.75.")


Criterio de Vocabulario: PORCENTAJE de palabras unicas = 100.0% (53107 palabras).
Vocabulario construido exitosamente al azar (semilla=26). Tamanio total: 53108 tokens.
Total de indices procesados: 916,200
Muestreo Negativo desactivado (Softmax Completa): Se omite el cálculo de la distribución unigrama P(w)^0.75.


## Sección 4: Entrenamiento del Modelo CBOW
Celda principal de entrenamiento. Utiliza el motor configurado (`"cupy"` o `"pytorch"`), la ventana $W$ y los parámetros de Muestreo Negativo.

In [4]:
tamanio_w = config.tamanio_ventana
tamanio_v = vocabulario.tamanio_vocabulario
nombre_modelo = f"modelo_w{tamanio_w}"

print(f"=== Iniciando Entrenamiento del Modelo: '{nombre_modelo}' (W = {tamanio_w}, Motor = {config.motor_computo}) ===")
print(f"Muestreo Negativo: {config.muestreo_negativo} | Cantidad Negativos (K): {config.cantidad_muestras_negativas}")

# Instanciar modelo CBOW con seleccion dinamica de motor
if config.motor_computo == "pytorch":
    modelo_cbow = ModeloCbowPyTorch(
        tamanio_vocabulario=tamanio_v,
        dimension_embedding=config.dimension_embedding,
        semilla_aleatoria=config.semilla_aleatoria,
        muestreo_negativo=config.muestreo_negativo,
        cantidad_muestras_negativas=config.cantidad_muestras_negativas,
        distribucion_unigrama=distribucion_unigrama,
    )
else:
    modelo_cbow = ModeloCbowCuPy(
        tamanio_vocabulario=tamanio_v,
        dimension_embedding=config.dimension_embedding,
        semilla_aleatoria=config.semilla_aleatoria,
        muestreo_negativo=config.muestreo_negativo,
        cantidad_muestras_negativas=config.cantidad_muestras_negativas,
        distribucion_unigrama=distribucion_unigrama,
    )

# Instanciar el entrenador
entrenador_cbow = EntrenadorCbow(
    modelo=modelo_cbow,
    tasa_aprendizaje=config.tasa_aprendizaje,
    directorio_respaldos=config.directorio_respaldos,
        hacer_respaldo=config.hacer_respaldo,
    frecuencia_respaldo=config.frecuencia_respaldo,
)

# Entrenar modelo
historial_perdida = entrenador_cbow.entrenar(
    indices_tokens=indices_corpus,
    tamanio_ventana=tamanio_w,
    tamanio_lote=config.tamanio_lote,
    cantidad_epocas=config.cantidad_epocas,
)

=== Iniciando Entrenamiento del Modelo: 'modelo_w4' (W = 4, Motor = pytorch) ===
Muestreo Negativo: False | Cantidad Negativos (K): 5

--- Iniciando Entrenamiento CBOW de Alta Velocidad (GPU) ---
Ventana (W): 4 | Batch: 2048 | Epocas Objetivo: 10 | Tasa: 0.2
Total de muestras: 916,192 distribuidas en 448 lotes pre-cargados en VRAM.


Epoca 1/10: 100%|█████████████████████████████████████████████████████████████| 448/448 [00:46<00:00,  9.72lote/s, Perdida=10.8801]


Epoca 1 finalizada. Perdida Promedio: 10.8801 | Tiempo: 46.11s (19,870.3 muestras/s)


Epoca 2/10: 100%|█████████████████████████████████████████████████████████████| 448/448 [00:47<00:00,  9.52lote/s, Perdida=10.8800]


Epoca 2 finalizada. Perdida Promedio: 10.8801 | Tiempo: 47.08s (19,459.1 muestras/s)


...

Epoca 8 finalizada. Perdida Promedio: 10.3607 | Tiempo: 47.13s (19,440.8 muestras/s)


Epoca 9/10: 100%|██████████████████████████████████████████████████████████████| 448/448 [00:48<00:00,  9.26lote/s, Perdida=9.4525]


Epoca 9 finalizada. Perdida Promedio: 9.6200 | Tiempo: 48.37s (18,942.1 muestras/s)


Epoca 10/10: 100%|█████████████████████████████████████████████████████████████| 448/448 [00:50<00:00,  8.87lote/s, Perdida=9.1764]


Epoca 10 finalizada. Perdida Promedio: 9.2250 | Tiempo: 50.54s (18,129.8 muestras/s)
Backup PyTorch guardado exitosamente en: respaldos/modelo_cbow_w4_epoca_10.npz


### Sección 4.1: Reanudar / Continuar el Entrenamiento por Épocas Adicionales
Reanuda el entrenamiento desde un archivo `.npz` respetando la configuración dinámica del motor y actualizando la variable global `modelo_cbow` para las evaluaciones posteriores.

In [ ]:
# --- REANUDACIÓN DE ENTRENAMIENTO --- 
ruta_backup = Path(config.directorio_respaldos) / f"modelo_cbow_w{config.tamanio_ventana}_epoca_10.npz"

if ruta_backup.exists():
    # 1. Instanciacion dinamica del modelo segun el motor configurado
    if config.motor_computo == "pytorch":
        modelo_reanudado = ModeloCbowPyTorch(
            tamanio_vocabulario=vocabulario.tamanio_vocabulario,
            dimension_embedding=config.dimension_embedding,
            muestreo_negativo=config.muestreo_negativo,
            cantidad_muestras_negativas=config.cantidad_muestras_negativas,
            distribucion_unigrama=distribucion_unigrama,
        )
    else:
        modelo_reanudado = ModeloCbowCuPy(
            tamanio_vocabulario=vocabulario.tamanio_vocabulario,
            dimension_embedding=config.dimension_embedding,
            muestreo_negativo=config.muestreo_negativo,
            cantidad_muestras_negativas=config.cantidad_muestras_negativas,
            distribucion_unigrama=distribucion_unigrama,
        )
    
    # 2. Cargar los pesos y el historial guardado
    epoca_guardada, historial_anterior = modelo_reanudado.cargar_modelo(ruta_backup)
    
    # 3. Sobrescribir la variable global modelo_cbow para unificar evaluacion
    modelo_cbow = modelo_reanudado
    
    # 4. Instanciar entrenador y entrenar N epocas adicionales
    entrenador_reanudado = EntrenadorCbow(
        modelo=modelo_cbow,
        tasa_aprendizaje=config.tasa_aprendizaje,
        directorio_respaldos=config.directorio_respaldos,
        frecuencia_respaldo=config.frecuencia_respaldo,
    )
    entrenador_reanudado.historial_perdida = list(historial_anterior)
    entrenador_reanudado.epoca_actual_cargada = epoca_guardada
    
    perdida_actualizada = entrenador_reanudado.entrenar_adicional(
        indices_tokens=indices_corpus,
        epocas_adicionales=5,
        tamanio_ventana=config.tamanio_ventana,
        tamanio_lote=config.tamanio_lote,
    )
else:
    print(f"El archivo {ruta_backup} no existe todavia en la carpeta respaldos/.")

## Sección 5: Evaluación de Similaridad de Palabras (Producto Interno vs Coseno)
Evaluamos la similaridad de una palabra objetivo ingresada por el usuario utilizando la variable unificada `modelo_cbow`.

In [ ]:
# Instanciar evaluador utilizando la variable global modelo_cbow
evaluador = EvaluadorSimilaridad(modelo_cbow, vocabulario)

# Palabra ingresada para la comparacion
palabra_evaluar = "hombre"
top_k = 10

print(f"=== Busqueda por PRODUCTO INTERNO para la palabra '{palabra_evaluar}' ===\n")
similares_pi = evaluador.buscar_palabras_similares(
    palabra_evaluar,
    top_k=top_k,
    tipo_similaridad="producto_interno",
)

posicion = 1
for par in similares_pi:
    p, s = par
    print(f"  {posicion:2d}. {p:<15} Puntaje: {s:.4f}")
    posicion += 1

print(f"\n=== Busqueda por SIMILARIDAD DE COSENO para la palabra '{palabra_evaluar}' ===\n")
similares_cos = evaluador.buscar_palabras_similares(
    palabra_evaluar,
    top_k=top_k,
    tipo_similaridad="coseno",
)

posicion = 1
for par in similares_cos:
    p, s = par
    print(f"  {posicion:2d}. {p:<15} Coseno: {s:.4f}")
    posicion += 1

## Sección 6: Comparación entre Dos Modelos o Resguardos (.npz)
Carga y compara las curvas de pérdida y similaridad de palabras entre dos archivos de resguardo `.npz` especificando sus rutas.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# --- RUTAS DE LOS DOS MODELOS A COMPARAR ---
ruta_modelo_1 = Path(config.directorio_respaldos) / "modelo_cbow_w4_epoca_10.npz"
ruta_modelo_2 = Path(config.directorio_respaldos) / "modelo_cbow_w5_epoca_10.npz"

print('=== Comparación de Modelos ===')
print(f'Modelo 1: {ruta_modelo_1}')
print(f'Modelo 2: {ruta_modelo_2}\n')

loss_m1 = []
loss_m2 = []

if ruta_modelo_1.exists():
    if config.motor_computo == 'pytorch':
        mod_1 = ModeloCbowPyTorch(tamanio_vocabulario=vocabulario.tamanio_vocabulario, dimension_embedding=config.dimension_embedding)
    else:
        mod_1 = ModeloCbowCuPy(tamanio_vocabulario=vocabulario.tamanio_vocabulario, dimension_embedding=config.dimension_embedding)
    ep_1, loss_m1 = mod_1.cargar_modelo(ruta_modelo_1)
else:
    print(f"El archivo {ruta_modelo_1} no existe todavía.")

if ruta_modelo_2.exists():
    if config.motor_computo == 'pytorch':
        mod_2 = ModeloCbowPyTorch(tamanio_vocabulario=vocabulario.tamanio_vocabulario, dimension_embedding=config.dimension_embedding)
    else:
        mod_2 = ModeloCbowCuPy(tamanio_vocabulario=vocabulario.tamanio_vocabulario, dimension_embedding=config.dimension_embedding)
    ep_2, loss_m2 = mod_2.cargar_modelo(ruta_modelo_2)
else:
    print(f"El archivo {ruta_modelo_2} no existe todavía.")

# Graficar curvas de perdida comparativas
if len(loss_m1) > 0 or len(loss_m2) > 0:
    sns.set_theme(style="whitegrid")
    plt.figure(figsize=(10, 5))
    
    if len(loss_m1) > 0:
        epocas_m1 = []
        for idx in range(1, len(loss_m1) + 1):
            epocas_m1.append(idx)
        plt.plot(epocas_m1, loss_m1, marker="o", color="#1f77b4", linewidth=2, label=f"Modelo 1 ({ruta_modelo_1.name})")
        
    if len(loss_m2) > 0:
        epocas_m2 = []
        for idx in range(1, len(loss_m2) + 1):
            epocas_m2.append(idx)
        plt.plot(epocas_m2, loss_m2, marker="s", color="#ff7f0e", linewidth=2, label=f"Modelo 2 ({ruta_modelo_2.name})")

    plt.title("Evolución Comparativa de Pérdida Promedio por Época", fontsize=14, fontweight="bold")
    plt.xlabel("Época", fontsize=12)
    plt.ylabel("Pérdida Promedio", fontsize=12)
    plt.legend(fontsize=11)
    plt.tight_layout()
    plt.show()

## Sección 7: Conclusiones y Registro en Bitácora

*(Esta sección se encuentra libre para completar con tus conclusiones finales y hallazgos principales tras la ejecución de los experimentos)*

- **Conclusión 1**:
- **Conclusión 2**:
- **Conclusión 3**: